# 03. Базовые модели и масштабирование

Мы выбрали четыре способа прогноза: последнее известное значение, среднее обучающей выборки, линейную регрессию Ridge и градиентный бустинг. Сильная связь соседних измерений делает прогноз по последнему наблюдению естественной отправной точкой; обучение ему не требуется. Прогноз среднего помогает оценить, что получится без учёта изменений режима, Ridge — проверить линейные зависимости, а бустинг — учесть более сложные связи.

Для Ridge отдельно проверили масштабирование числовых признаков, чтобы выяснить, влияет ли оно на качество в этой задаче.

In [1]:
import pandas as pd
from IPython.display import display
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import RobustScaler, StandardScaler

from src.evaluation import RegressionExperiment
from src.features import ALL_FEATURES, LAG_FEATURES, prepare_feature_dataset
from src.pipelines import build_boosting_regressor, build_regression_pipeline

feature_dataset = prepare_feature_dataset()
experiment = RegressionExperiment(feature_dataset)

## Условия сравнения

Для сопоставимости результатов использовали одинаковые временные разбиения и набор признаков. Час, день недели и тип дня закодировали методом one-hot (кодирование категорий отдельными бинарными признаками). Для Ridge зафиксировали `alpha = 1` и сравнили StandardScaler (стандартизация по среднему и стандартному отклонению), RobustScaler (масштабирование по медиане и межквартильному размаху) и вариант без масштабирования. Помимо ошибок, оценили число ненулевых коэффициентов и их максимальный абсолютный размер. Предобработку обучали заново на каждом обучающем периоде, чтобы не использовать сведения из будущего.

In [2]:
experiment.evaluate_regressor("Последний интервал", LAG_FEATURES[:1])
dummy_regressor = build_regression_pipeline(ALL_FEATURES, DummyRegressor())
experiment.evaluate_regressor(
    "Среднее обучающей выборки", ALL_FEATURES, dummy_regressor
)
for regressor_name, scaler in [
    ("Ridge · без скейлера", "passthrough"),
    ("Ridge · StandardScaler", StandardScaler()),
    ("Ridge · RobustScaler", RobustScaler()),
]:
    ridge_regressor = build_regression_pipeline(
        ALL_FEATURES, Ridge(alpha=1, solver="svd"), scaler
    )
    experiment.evaluate_regressor(regressor_name, ALL_FEATURES, ridge_regressor)
boosting_regressor = build_boosting_regressor(ALL_FEATURES)
experiment.evaluate_regressor(
    "Бустинг · лаги и календарь", ALL_FEATURES, boosting_regressor
)
display(experiment.summarize_results().round(3))
coefficient_diagnostics = pd.DataFrame(experiment.coefficient_diagnostics)
display(
    coefficient_diagnostics.groupby("regressor", sort=False)[
        ["coefficient_count", "nonzero_count", "maximum_absolute_coefficient"]
    ]
    .mean()
    .round(3)
)
experiment.save_results("baselines")

,MAE,MAE_std,RMSE,RMSE_std,R2,R2_std
regressor,,,,,,
Последний интервал,5.835,0.444,13.079,0.875,0.828,0.018
Среднее обучающей выборки,28.776,1.492,32.268,1.328,-0.047,0.049
Ridge · без скейлера,6.828,0.575,11.463,0.828,0.868,0.015
Ridge · StandardScaler,6.828,0.575,11.463,0.828,0.868,0.015
Ridge · RobustScaler,6.829,0.576,11.464,0.828,0.868,0.015
Бустинг · лаги и календарь,4.808,0.684,9.773,1.097,0.903,0.018


,coefficient_count,nonzero_count,maximum_absolute_coefficient
regressor,,,
Ridge · без скейлера,38.0,38.0,12.234
Ridge · StandardScaler,38.0,38.0,27.978
Ridge · RobustScaler,38.0,38.0,41.927


## Результаты

Прогноз по последнему наблюдению дал MAE (средняя абсолютная ошибка) 5,835 кВт·ч, а прогноз среднего — 28,776 кВт·ч. Такой разрыв показывает, насколько полезна информация о недавнем потреблении. У Ridge ошибка составила около 6,83 кВт·ч во всех трёх вариантах масштабирования, то есть линейная модель не улучшила прогноз по последнему наблюдению.

Во всех вариантах Ridge получилось 38 ненулевых коэффициентов. Максимальный модуль коэффициента изменился с 12,2 без масштабирования до 28,0 со StandardScaler и 41,9 с RobustScaler. Поскольку эти величины зависят от масштаба признаков, их различия нельзя интерпретировать как изменение значимости. Существенного улучшения MAE масштабирование не дало.

Наименьшую ошибку дал бустинг — 4,808 кВт·ч. Поэтому дальше исследуем, как на его качество влияют состав признаков и подбор гиперпараметров.